<table><tr>
<td width="76"><div align="center" style="font-size:44px">🎯</div></td>
<td><h1 style="margin:0">LAB 4 · Tu primera regla YARA</h1>
<b>Máster en Cyber Threat Intelligence · Módulo 06 · Sesión 44 — Análisis estático de malware</b><br>
⏱️ <b>12 minutos</b> &nbsp;·&nbsp; 🎯 Convertir lo que has encontrado hoy en una detección que funcione mañana</td>
</tr></table>

---

## La situación

Has hecho un buen trabajo: sabes que `curriculum_vitae_2024.pdf` es njRAT y sabes cuál es su C2.

Pero mañana el atacante recompila y **cambia el hash**. Tu IOC deja de servir.

Tu jefa te pide lo último del día:

> *"Escríbeme una detección que siga funcionando cuando cambien el fichero."*

Para eso está **YARA**: en vez de identificar *un fichero*, describe **un patrón**.
Es el idioma estándar de la industria para esto, y se usa en todas partes: antivirus, EDR,
VirusTotal, escaneo de correo, análisis de memoria…

### Lo que tienes que entregar
1. 📜 **Tu regla YARA**, pegada en el chat
2. 🔢 **Cuántas muestras detecta** de las 9
3. 🚫 **Cero falsos positivos** (esto es lo difícil, ya verás)

---
## 🔧 Preparación · ejecuta esta celda SIEMPRE

**Cada laboratorio es un cuaderno distinto y arranca en una máquina nueva.**
Aunque vengas del laboratorio anterior, aquí no hay nada instalado y las muestras
todavía no están. No se comparte nada entre cuadernos.

Pulsa ▶️ en la celda de abajo y espera unos **40 segundos**. Solo hay que hacerlo
una vez por laboratorio.

> 📦 La segunda celda, **PLAN B**, solo hace falta si la primera no consigue las
> muestras. Si la primera termina con el listado de ficheros, ignórala y sigue.


In [ ]:
#@title ▶️ EJECUTA ESTA CELDA (botón ▶ a la izquierda) y espera ~40 segundos { display-mode: "form" }

#@markdown ---
#@markdown **No hace falta que entiendas este código todavía.** Solo prepara el laboratorio:
#@markdown instala las herramientas, descarga las muestras y las descomprime.
#@markdown ---

SAMPLES_URL = "https://github.com/jstnk9/kschool_ejercicios/raw/main/analisis_estatico/muestras_kschool.zip" #@param {type:"string"}
PASSWORD    = "infected" #@param {type:"string"}

import os, glob, subprocess

def _sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)

print("1/4  Instalando herramientas de análisis...")
_sh("pip install -q pyzipper pefile oletools yara-python py-tlsh")
print("     ✔ pyzipper, pefile, oletools, yara-python, tlsh")

print("2/4  Consiguiendo las muestras...")
DEST = "/content/muestras_kschool.zip"

if not SAMPLES_URL.strip():
    # Sin URL configurada: las muestras se suben a mano con la celda de abajo.
    print("     ℹ  Este cuaderno no trae URL de descarga.")
    print("        Ve a la celda de abajo, 'PLAN B', y sube el fichero")
    print("        muestras_kschool.zip que te ha pasado el profesor.")
    print("        Es un paso normal: tarda 10 segundos.")
    ok_zip = False
else:
    url = SAMPLES_URL.strip()

    # GitHub sirve DOS urls distintas para el mismo fichero:
    #   .../blob/...  -> la pagina web que lo muestra  (HTML)
    #   .../raw/...   -> el fichero de verdad
    # Si te has copiado la de la barra del navegador, la arreglamos aqui.
    if "github.com" in url and "/blob/" in url:
        url = url.replace("/blob/", "/raw/")
        print("     ℹ  URL de GitHub corregida: /blob/ -> /raw/")
    url = url.replace("?raw=true", "").replace("?raw=1", "")

    if os.path.exists(DEST):
        os.remove(DEST)      # por si un intento anterior dejo un fichero malo

    if "drive.google.com" in url:
        _sh("pip install -q gdown")
        _sh("gdown --fuzzy '" + url + "' -O " + DEST)
    else:
        _sh("wget -q --no-check-certificate '" + url + "' -O " + DEST)

    # Comprobamos que lo descargado es DE VERDAD un ZIP mirando sus primeros
    # bytes. Que es, mira tu por donde, justo lo que vas a aprender hoy:
    # un ZIP siempre empieza por 50 4B 03 04, o sea "PK".
    cabecera = open(DEST, "rb").read(4) if os.path.exists(DEST) else b""
    ok_zip = cabecera == b"PK\x03\x04"

    if ok_zip:
        print("     ✔ Descargado (" + str(os.path.getsize(DEST)//1024) + " KB, empieza por 'PK' ✔)")
    else:
        print("     ✖ Lo que he descargado NO es un ZIP.")
        print("        Empieza por los bytes " + (cabecera.hex() or "(nada)") +
              " y un ZIP empieza siempre por 504b0304.")
        if b"<" in cabecera or b"\n" in cabecera:
            print("")
            print("        Parece una pagina HTML. Lo tipico: la URL apunta a la PAGINA")
            print("        de GitHub y no al fichero. Fijate en la diferencia:")
            print("           .../blob/main/...  <- pagina web   ✖")
            print("           .../raw/main/...   <- el fichero   ✔")
            print("        Pulsa el boton 'Raw' en GitHub y copia esa URL.")
        print("")
        print("        Alternativa: usa la celda de abajo, 'PLAN B'.")

print("3/4  Descomprimiendo (contraseña: infected)...")
import pyzipper
if ok_zip:
    try:
        with pyzipper.AESZipFile(DEST) as z:
            z.setpassword(PASSWORD.encode())
            z.extractall("/content/")
        print("     ✔ Descomprimido en /content/muestras/")
    except Exception as e:
        print("     ✖ Error al descomprimir:", e)

print("4/4  Comprobando el laboratorio...")
MUESTRAS = "/content/muestras"
ficheros = sorted(f for f in glob.glob(MUESTRAS + "/*") if not f.endswith("LEEME.txt"))
if len(ficheros) >= 9:
    print("     ✔ " + str(len(ficheros)) + " muestras listas")
    print("")
    print("=" * 52)
    print("   LABORATORIO LISTO  ✅")
    print("=" * 52)
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("     ✖ Solo encuentro " + str(len(ficheros)) + " ficheros. Usa la celda 'PLAN B' de abajo.")

print("""
⚠️  RECUERDA: esto son muestras REALES de malware.
    Estás dentro de una máquina virtual de Google que se destruye al cerrar.
    NO descargues estos ficheros a tu ordenador. NO los ejecutes.
    Hoy solo vamos a MIRARLOS, que es justo de lo que va el análisis estático.
""")

In [ ]:
#@title 📦 PLAN B — solo si la celda de arriba no ha conseguido las muestras { display-mode: "form" }
import glob, os

# Esta celda se puede ejecutar sola, sin haber pasado por la de arriba,
# asi que se instala ella misma lo que necesita.
try:
    import pyzipper
except ImportError:
    print("Instalando pyzipper (5 segundos)...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyzipper"], check=False)
    import pyzipper

ZIP_YA_SUBIDO = "/content/muestras_kschool.zip"

if os.path.exists(ZIP_YA_SUBIDO):
    # ya lo habias subido con el panel de Archivos de la izquierda
    print("Encontrado", ZIP_YA_SUBIDO, "- no hace falta que lo subas otra vez.")
    nombres = [ZIP_YA_SUBIDO]
else:
    from google.colab import files
    print("Pulsa en 'Elegir archivos' y selecciona muestras_kschool.zip")
    nombres = list(files.upload())

for nombre in nombres:
    try:
        with pyzipper.AESZipFile(nombre) as z:
            z.setpassword(b"infected")
            z.extractall("/content/")
    except Exception as e:
        print("✖ No he podido abrir", nombre, "->", e)

ficheros = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))
if ficheros:
    print("")
    print("✔ " + str(len(ficheros)) + " muestras listas en /content/muestras/")
    for f in ficheros:
        print("   {:>9,} bytes   {}".format(os.path.getsize(f), os.path.basename(f)))
else:
    print("✖ Sigo sin ver las muestras. Avisa en el chat.")

---
# Paso 1 · Anatomía de una regla

Una regla YARA tiene tres partes y se lee casi como una frase en inglés:

```yara
rule NOMBRE_DE_LA_REGLA
{
    meta:                                  // 1. Documentación. No afecta a la detección.
        autor       = "tu nombre"
        descripcion = "qué detecta y por qué"

    strings:                               // 2. QUÉ buscar
        $a = "texto que buscamos"
        $b = { 4D 5A }                     //    también valen bytes en hexadecimal

    condition:                             // 3. CUÁNDO se considera detectado
        $a and $b
}
```

En `condition` puedes escribir cosas como:

| Condición | Significa |
|---|---|
| `$a` | Aparece `$a` |
| `$a and $b` | Aparecen los dos |
| `2 of ($s*)` | Al menos 2 de las strings que empiezan por `$s` |
| `all of them` | Todas |
| `filesize < 2MB` | El fichero mide menos de 2 MB |
| `uint16(0) == 0x5A4D` | Los 2 primeros bytes son `MZ` → **es un ejecutable de Windows** |

Y los modificadores que van detrás de cada string:

| Modificador | Para qué |
|---|---|
| `ascii` | Buscar en texto normal (por defecto) |
| `wide` | Buscar en UTF-16 — **¿te suena del LAB 2?** |
| `nocase` | Ignorar mayúsculas/minúsculas |
| `fullword` | Solo si es una palabra completa |

> 💡 `ascii wide` juntos = busca en los dos sitios. **Ponlo casi siempre.**
> Acuérdate de que el C2 del LAB 2 solo estaba en wide.

Vamos a probar una regla ya escrita.

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import yara, glob, os

MUESTRAS = sorted(f for f in glob.glob("/content/muestras/*") if not f.endswith("LEEME.txt"))

REGLA_EJEMPLO = """
rule Detecta_Ejecutable_Windows
{
    meta:
        autor       = "KSchool"
        descripcion = "Lo mas basico: detecta cualquier PE de Windows"

    condition:
        uint16(0) == 0x5A4D          // los 2 primeros bytes son 'MZ'
}
"""

reglas = yara.compile(source=REGLA_EJEMPLO)

print("Escaneando las 9 muestras...")
print()
for f in MUESTRAS:
    coincide = reglas.match(f)
    print("   {}  {}".format("🎯 DETECTADO" if coincide else "·  limpio    ",
                             os.path.basename(f)))

Funciona, pero es inútil: detecta **cualquier** ejecutable de Windows, incluido el Bloc de notas.

Una regla útil tiene que ser **específica**. Vamos a usar lo que encontramos en el LAB 2.

---
# Paso 2 · Una regla de verdad, con lo que encontramos hoy

Del LAB 2 nos llevamos estas pistas de njRAT:

- El C2 codificado: `aGFraW0z*i5kZG5zLm5ldA!!`
- El alias del atacante: `FRANSESCO`
- Los nombres de función en portugués: `EnviarDadosConexaooo`, `EnviarPermitirFormJanelas`…

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import yara, glob, os

# se recalculan por si ejecutas esta celda suelta
MUESTRAS = sorted(f for f in glob.glob("/content/muestras/*")
                  if not f.endswith("LEEME.txt"))
LIMPIOS  = sorted(glob.glob("/content/limpios/*"))

REGLA_NJRAT = """
rule MALW_njRAT_Fransesco
{
    meta:
        autor       = "KSchool - Master CTI"
        descripcion = "njRAT, campana con el alias FRANSESCO y funciones en portugues"
        referencia  = "3b7c80a670ed7981e02530ca4fc4ff52e46ebe19e6ddaf3fde249d25918da77b"

    strings:
        $c2   = "aGFraW0z*i5kZG5zLm5ldA!!"      ascii wide
        $alias = "FRANSESCO"                    ascii wide

        $f1 = "EnviarPermitirFormJanelas"       ascii wide
        $f2 = "EnviarDadosConexaooo"            ascii wide
        $f3 = "EnviarResultadoGerenciadorrr"    ascii wide

    condition:
        uint16(0) == 0x5A4D          // tiene que ser un ejecutable de Windows
        and filesize < 5MB           // njRAT es pequeno; descarta cosas raras
        and 2 of ($f*)               // al menos 2 de los nombres de funcion
        and ($c2 or $alias)          // y ademas el C2 o el alias
}
"""

reglas = yara.compile(source=REGLA_NJRAT)

for f in MUESTRAS:
    m = reglas.match(f)
    if m:
        print("🎯 DETECTADO:", os.path.basename(f))
        for coincidencia in m:
            for s in coincidencia.strings:
                for instancia in s.instances:
                    print("      {} en offset {}".format(s.identifier, hex(instancia.offset)))

### Lee la condición otra vez, despacio:

```
uint16(0) == 0x5A4D  and  filesize < 5MB  and  2 of ($f*)  and  ($c2 or $alias)
```

Eso está diciendo: *"es un ejecutable de Windows, pequeño, con al menos dos de los nombres
de función característicos de esta familia, y además el C2 o el alias del atacante"*.

**Lo importante es que esa regla sobrevive a que cambie el hash.** El atacante puede
recompilar mil veces: mientras el código siga siendo el mismo njRAT, la regla lo pilla.

> Así es como funcionan de verdad las reglas de caza en VirusTotal Hunting, en un EDR
> o en el correo corporativo: no buscan ficheros, buscan **familias**.

---
## 🧩 TU TURNO (5 minutos) — Escribe la tuya

Ahora te toca. **Escribe una regla que detecte troyanos que se hacen persistentes.**

No una familia concreta: un **comportamiento**. Eso se llama una regla *genérica* o *de
comportamiento*, y es lo que usas cuando no sabes a qué te enfrentas.

Las pistas del LAB 2 que te valen:

```
Software\Microsoft\Windows\CurrentVersion\Run
schtasks /create
netsh firewall add allowedprogram
cmd.exe /c ping 0 -n 2 & del "
```

**Rellena los huecos de abajo y ejecuta.** No te preocupes por acertar a la primera.

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import yara, glob, os

# se recalculan por si ejecutas esta celda suelta
MUESTRAS = sorted(f for f in glob.glob("/content/muestras/*")
                  if not f.endswith("LEEME.txt"))
LIMPIOS  = sorted(glob.glob("/content/limpios/*"))

MI_REGLA = """
rule Mi_Regla_Persistencia
{
    meta:
        autor       = "PON TU NOMBRE"
        descripcion = "Detecta ejecutables que intentan hacerse persistentes"

    strings:
        $p1 = "Software\\\\Microsoft\\\\Windows\\\\CurrentVersion\\\\Run"  ascii wide
        $p2 = "schtasks"                                        ascii wide

        // 👇 ANADE AQUI 2 STRINGS MAS (mira las pistas de arriba)
        $p3 = "CAMBIAME"                                        ascii wide
        $p4 = "CAMBIAME"                                        ascii wide

    condition:
        // 👇 ESCRIBE AQUI TU CONDICION
        //    empieza con algo sencillo, por ejemplo:   2 of them
        2 of them
}
"""

reglas_mias = yara.compile(source=MI_REGLA)

print("RESULTADO DE TU REGLA")
print("=" * 60)
detectados = 0
for f in MUESTRAS:
    if reglas_mias.match(f):
        detectados += 1
        print("   🎯", os.path.basename(f))
print()
print("Detecta {} de {} muestras".format(detectados, len(MUESTRAS)))

In [ ]:
#@title ✅ Solución del TU TURNO — ábrela solo si te has atascado (doble clic para ver el código)
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import yara, glob, os

# se recalculan por si ejecutas esta celda suelta
MUESTRAS = sorted(f for f in glob.glob("/content/muestras/*")
                  if not f.endswith("LEEME.txt"))
LIMPIOS  = sorted(glob.glob("/content/limpios/*"))

MI_REGLA = """
rule Mi_Regla_Persistencia
{
    meta:
        autor       = "KSchool"
        descripcion = "Detecta ejecutables que intentan hacerse persistentes y bajar defensas"

    strings:
        $p1 = "Software\\\\Microsoft\\\\Windows\\\\CurrentVersion\\\\Run"  ascii wide
        $p2 = "schtasks"                                        ascii wide
        $p3 = "netsh firewall add allowedprogram"               ascii wide
        $p4 = "cmd.exe /c ping 0 -n 2 & del"                    ascii wide

    condition:
        2 of them
}
"""

reglas_mias = yara.compile(source=MI_REGLA)

print("RESULTADO")
print("=" * 60)
for f in MUESTRAS:
    m = reglas_mias.match(f)
    if m:
        cuales = [s.identifier for s in m[0].strings]
        print("   🎯 {:<42} {}".format(os.path.basename(f), " ".join(cuales)))

print()
print("Detecta njRAT. Parece que ya está, ¿no?")
print("Pues no. Sigue bajando: falta la parte que de verdad importa. 👇")

---
# Paso 3 · El giro: los falsos positivos

Tu regla detecta el malware. Perfecto. Pero…

> **Una regla no se juzga por lo que detecta. Se juzga por lo que detecta DE MÁS.**

En un SOC real, tu regla no se ejecuta contra 9 ficheros: se ejecuta contra **millones**.
Si genera un 0,1% de falsos positivos, has enterrado a tu equipo en alertas inútiles y
lo primero que harán será desactivarla.

Vamos a comprobarlo. La celda de abajo **crea tres ficheros completamente inofensivos**:
son los que tendría cualquier informático en su carpeta de trabajo.

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import glob, os
os.makedirs("/content/limpios", exist_ok=True)

# 1. Un manual interno de soporte IT
open("/content/limpios/manual_soporte_IT.txt", "w", encoding="utf-8").write("""
MANUAL INTERNO - DEPARTAMENTO DE SISTEMAS

Como revisar el arranque de un equipo:
  1. Abrir regedit y navegar a Software\\Microsoft\\Windows\\CurrentVersion\\Run
  2. Comprobar que no hay entradas desconocidas
  3. Revisar las tareas programadas con: schtasks /query
  4. Si hay que abrir un puerto: netsh firewall add allowedprogram ...

Para diagnosticar la red se puede usar cmd.exe /c ping.
""")

# 2. El script de despliegue legitimo de la propia empresa
open("/content/limpios/desplegar_agente.bat", "w", encoding="utf-8").write(
    "@echo off\r\n"
    "REM Despliegue del agente corporativo de inventario\r\n"
    "reg add HKLM\\Software\\Microsoft\\Windows\\CurrentVersion\\Run /v Inventario /d agente.exe\r\n"
    "schtasks /create /sc daily /tn InventarioDiario /tr agente.exe\r\n"
    "netsh firewall add allowedprogram agente.exe Inventario ENABLE\r\n"
)

# 3. Un informe de amenazas... que HABLA del malware (¡clásico!)
open("/content/limpios/informe_amenazas_Q3.txt", "w", encoding="utf-8").write("""
INFORME DE INTELIGENCIA DE AMENAZAS - Q3

La familia njRAT persiste mediante Software\\Microsoft\\Windows\\CurrentVersion\\Run
y crea tareas con schtasks. Tambien ejecuta netsh firewall add allowedprogram
para permitirse trafico saliente. El alias observado es FRANSESCO.
""")

LIMPIOS = sorted(glob.glob("/content/limpios/*"))

print("He creado 3 ficheros 100% inofensivos:")
for f in LIMPIOS:
    print("   ✅", os.path.basename(f))
print()
print("Ahora pasamos TU regla sobre ellos...")
print("=" * 60)

falsos = 0
for f in LIMPIOS:
    if reglas_mias.match(f):
        falsos += 1
        print("   🔴 FALSO POSITIVO:", os.path.basename(f))

if falsos:
    print()
    print("😬 {} falsos positivos de 3.".format(falsos))
    print("   Acabas de acusar de malware a un manual de soporte,")
    print("   al script de despliegue de tu propia empresa")
    print("   y a un informe de amenazas que solo HABLA del malware.")
else:
    print("   ✅ Ningun falso positivo. Bien hecho.")

### 😅 Ese tercer fichero es la mejor broma del oficio

Un **informe de inteligencia de amenazas** que describe a njRAT contiene, por narices, todos
los IOCs de njRAT. Así que una regla mal hecha **detecta como malware a la documentación
que describe el malware**.

No es un chiste: pasa constantemente en producción. Los informes en PDF, los tickets del SOC,
los correos entre analistas… todos llevan IOCs dentro.

## 🧩 TU TURNO (3 minutos) — Arréglala

Tienes que hacer que la regla siga detectando las muestras **y deje en paz a los tres limpios**.

> 💡 **La pista es una sola línea.** Fíjate en qué tienen en común los tres falsos positivos
> y qué tienen en común las muestras de verdad.
>
> Los limpios son `.txt` y `.bat`: **texto**. Las muestras son **ejecutables**.
> ¿Cómo distinguías tú un ejecutable en el LAB 1?

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import yara, glob, os

# se recalculan por si ejecutas esta celda suelta
MUESTRAS = sorted(f for f in glob.glob("/content/muestras/*")
                  if not f.endswith("LEEME.txt"))
LIMPIOS  = sorted(glob.glob("/content/limpios/*"))

REGLA_ARREGLADA = """
rule Mi_Regla_Persistencia_v2
{
    strings:
        $p1 = "Software\\\\Microsoft\\\\Windows\\\\CurrentVersion\\\\Run"  ascii wide
        $p2 = "schtasks"                                        ascii wide
        $p3 = "netsh firewall add allowedprogram"               ascii wide
        $p4 = "cmd.exe /c ping 0 -n 2 & del"                    ascii wide

    condition:
        // 👇 ANADE AQUI LA LINEA QUE ARREGLA LOS FALSOS POSITIVOS
        2 of them
}
"""

reglas_v2 = yara.compile(source=REGLA_ARREGLADA)

print("MUESTRAS (aqui SI queremos detecciones)")
for f in MUESTRAS:
    if reglas_v2.match(f):
        print("   🎯", os.path.basename(f))

print()
print("LIMPIOS (aqui NO queremos ninguna)")
fp = [f for f in LIMPIOS if reglas_v2.match(f)]
for f in fp:
    print("   🔴", os.path.basename(f))
print("   ✅ ninguno" if not fp else "   😬 todavia hay {}".format(len(fp)))

In [ ]:
#@title ✅ Solución — la línea mágica — ábrela solo si te has atascado (doble clic para ver el código)
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import yara, glob, os

# se recalculan por si ejecutas esta celda suelta
MUESTRAS = sorted(f for f in glob.glob("/content/muestras/*")
                  if not f.endswith("LEEME.txt"))
LIMPIOS  = sorted(glob.glob("/content/limpios/*"))

REGLA_ARREGLADA = """
rule Mi_Regla_Persistencia_v2
{
    meta:
        autor       = "KSchool"
        descripcion = "Persistencia + evasion, solo en ejecutables de Windows"

    strings:
        $p1 = "Software\\\\Microsoft\\\\Windows\\\\CurrentVersion\\\\Run"  ascii wide
        $p2 = "schtasks"                                        ascii wide
        $p3 = "netsh firewall add allowedprogram"               ascii wide
        $p4 = "cmd.exe /c ping 0 -n 2 & del"                    ascii wide

    condition:
        uint16(0) == 0x5A4D          // <-- LA LINEA MAGICA: solo ejecutables de Windows
        and filesize < 10MB
        and 2 of them
}
"""

reglas_v2 = yara.compile(source=REGLA_ARREGLADA)

print("MUESTRAS")
for f in MUESTRAS:
    if reglas_v2.match(f):
        print("   🎯", os.path.basename(f))

print()
print("LIMPIOS")
fp = [f for f in LIMPIOS if reglas_v2.match(f)]
print("   ✅ Ningun falso positivo" if not fp else "   🔴 " + str(fp))

print()
print("=" * 66)
print("LA LECCION MAS IMPORTANTE DEL DIA:")
print("=" * 66)
print("""
  uint16(0) == 0x5A4D

  Una sola linea. No cambia NADA de lo que detectas,
  y elimina TODOS los falsos positivos.

  Eso es ingenieria de deteccion: no es escribir muchas reglas,
  es acotar bien las que escribes.

  Regla practica: empieza SIEMPRE la condicion anclando el TIPO DE FICHERO
  y el TAMANO. Es gratis y te ahorra el 90% de los falsos positivos.
""")

---
# 🏆 RETO final · Detecta cualquier cosa empaquetada con UPX

En el LAB 1 viste que UPX deja sus secciones llamadas `UPX0` y `UPX1`.

Escribe una regla que detecte **cualquier** ejecutable empaquetado con UPX, sin importar
qué malware lleve dentro. Debería detectar 3 de nuestras muestras.

In [ ]:
# Dependencias de esta celda (por si la ejecutas sin pasar por la de arriba)
import importlib, subprocess, sys
for _m, _p in [("yara", "yara-python")]:
    try:
        importlib.import_module(_m)
    except ImportError:
        print('Instalando ' + _p + '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p],
                       check=False)

import yara, glob, os

# se recalculan por si ejecutas esta celda suelta
MUESTRAS = sorted(f for f in glob.glob("/content/muestras/*")
                  if not f.endswith("LEEME.txt"))
LIMPIOS  = sorted(glob.glob("/content/limpios/*"))

REGLA_UPX = """
rule Empaquetado_Con_UPX
{
    meta:
        descripcion = "Cualquier PE empaquetado con UPX"

    strings:
        $s0 = "UPX0"
        $s1 = "UPX1"
        $s2 = "UPX!"

    condition:
        uint16(0) == 0x5A4D and 2 of them
}
"""

r_upx = yara.compile(source=REGLA_UPX)
for f in MUESTRAS + LIMPIOS:
    if r_upx.match(f):
        print("   📦 EMPAQUETADO:", os.path.basename(f))

### ⚠️ Y ahora la pregunta incómoda

Esa regla funciona. Pero **UPX es software legítimo**: hay programas normales, instaladores y
juegos que lo usan para ocupar menos.

Así que esta regla **no dice "esto es malware"**. Dice **"esto está empaquetado"**.

Y eso, en un SOC, se usa de otra manera: no como alerta, sino como **señal de contexto**
que sube la prioridad de lo que ya sospechabas. Un ejecutable desconocido, descargado de
internet, sin firma digital **y además empaquetado** → eso sí es una alerta.

> En detección casi nunca hay una única señal que diga "malo". Hay señales que suman.
> Confundir *"indicio"* con *"veredicto"* es el error más caro del oficio.

---
# ✅ Para la puesta en común

1. 📜 Pega **tu regla** en el chat
2. 🔢 Detecta **___ de 9** muestras
3. 🚫 Falsos positivos: **___ de 3**

---

## 🧰 Lo que te llevas de hoy

En dos horas has hecho, sin ejecutar una sola muestra:

| | |
|---|---|
| 🏷️ | Identificar el tipo real de un fichero por sus **magic bytes** |
| 🔢 | Calcular **hashes** y consultarlos en VirusTotal |
| 📦 | Detectar un **packer** por su entropía, sus secciones y sus imports |
| 🕵️ | Extraer **strings ASCII y wide** y sacar IOCs de ahí |
| 🔓 | Decodificar **Base64** y deshacer una **ofuscación** de un atacante |
| 📎 | Destripar un **maldoc** de Office y dos **PDF** maliciosos |
| 🎯 | Escribir **reglas YARA** y — más importante — no llenarlas de falsos positivos |

Eso es el flujo de trabajo real de un analista haciendo triaje. Con herramientas distintas,
más bonitas y más rápidas, pero **exactamente estos pasos**.

> 📓 Tienes un notebook extra, **BONUS**, con lo que no ha cabido: packers a fondo,
> fuzzy hashing, por qué el imphash falla con .NET y cómo pivotar en VirusTotal.
> Guárdate una copia en tu Drive antes de cerrar la pestaña.